# Modul 2 · Tabellen mit pandas
**Data Science 1** · linkit courses

Die Werksleitung der Nordwerk GmbH gibt dir zwei Jahre Werksdaten als fünf CSV-Dateien und will wissen, wie viel Ausschuss die vier Linien erzeugen, wo Stillstand entsteht und wie sich die Wochenleistung entwickelt.

**Lernziele**
1. Tabellen laden und ihre Datentypen prüfen
2. Zeilen mit Masken filtern
3. mit `groupby` aggregieren, auch Quoten
4. Tabellen über Schlüssel verknüpfen und die Zeilenzahl kontrollieren
5. zwischen breitem und langem Format umformen
6. mit Zeitstempeln rechnen

**So arbeitest du**
- Das Notebook läuft auf der Kursseite direkt im Browser, ohne Installation. Alternativ öffnest du es in Google Colab und speicherst dort zuerst eine eigene Kopie (*Datei → Kopie in Drive speichern*).
- Führe die Zellen von oben nach unten mit **Shift + Enter** aus.
- Stellen mit `# TODO` füllst du selbst aus, meist mit ein bis fünf Zeilen. Darunter prüft ein Selbsttest dein Ergebnis. Ein ❌ ist nur ein Hinweis, dass etwas noch nicht stimmt, das Notebook läuft trotzdem weiter.
- Zellen, die ein Ergebnis aus einem TODO brauchen, melden „⏳ Erst TODO n lösen.“, solange es fehlt.
- Mindestziel sind TODO 1 bis 5 und TODO 7 bis 10. TODO 6, TODO 11 und die Bonusaufgaben vertiefen.
- Jeder Abschnitt gehört zu einer Lerneinheit und zum Abschnitt mit derselben Nummer im Skript. Hilfen zu jedem TODO und typische Fehler findest du auf der Kursseite.

In [ ]:
# Hilfsfunktion für die Selbsttests: einfach ausführen, nicht ändern.
import math

def check(name, test):
    """Führt einen Test aus und zeigt ✅ oder ❌. Fehler im Test gelten als „noch nicht gelöst“."""
    try:
        ok = bool(test())
    except Exception as e:
        print(f"❌ {name}: noch nicht gelöst ({type(e).__name__}: {e})")
        return
    print(f"✅ {name}" if ok else f"❌ {name}: Ergebnis stimmt noch nicht")

print("Bereit.")

In [ ]:
# Daten laden: Auf der Kursseite liegen die Dateien schon bereit, in Colab holt diese Zelle sie von GitHub.
import os, urllib.request
DATEN = ["produktion.csv", "stoerungen.csv", "maschinen.csv", "linien.csv", "wochenbericht_breit.csv"]
QUELLE = "https://raw.githubusercontent.com/juliankunz18/linkit-courses/main/data-science-1/daten/"
os.makedirs("daten", exist_ok=True)
for datei in DATEN:
    if not os.path.exists("daten/" + datei):
        urllib.request.urlretrieve(QUELLE + datei, "daten/" + datei)
print("Daten bereit:", ", ".join(sorted(os.listdir("daten"))))

## 1 · Von der Liste zum Array: NumPy und Vektorisierung
*Lerneinheit 1 · Skript Abschnitt 1*

In Modul 1 war eine Tabelle eine Liste von Dictionaries, und jede Auswertung war eine Schleife. Die nächste Zelle liest `produktion.csv` genau so ein und berechnet die Ausschussquote $q_i = a_i / s_i$ jeder Schicht, also Ausschuss $a_i$ geteilt durch gefertigte Stück $s_i$. Achte darauf, dass jeder Wert als Text ankommt und erst mit `int(...)` zur Zahl wird.

In [ ]:
import csv

with open("daten/produktion.csv", encoding="utf-8") as f:
    zeilen = list(csv.DictReader(f))            # Liste von Dictionaries wie in Modul 1

print(len(zeilen), "Zeilen. Die erste:")
print(zeilen[0])

quoten_liste = []                               # Modul-1-Stil: Akkumulator in einer Schleife
for z in zeilen:
    quoten_liste.append(int(z["ausschuss_stueck"]) / int(z["ist_stueck"]))
print("Die ersten drei Quoten:", [round(q, 4) for q in quoten_liste[:3]])

Ein **NumPy-Array** speichert Werte eines einzigen Datentyps. Rechenoperationen wirken elementweise auf das ganze Array, ohne dass du eine Schleife schreibst. Das heißt **Vektorisierung**. Ein Vergleich wie `ist > 300` liefert ein Array aus `True` und `False`, eine **boolesche Maske**. In eckigen Klammern wählt sie Einträge aus, und weil `True` als 1 zählt, ist `maske.sum()` die Anzahl und `maske.mean()` der Anteil.

Im Browser zeigt `dtype` für Ganzzahlen `int32`, in Colab `int64`. Der Browser rechnet in einer 32-Bit-Umgebung, für Stückzahlen spielt das keine Rolle.

In [ ]:
import numpy as np

# Die fünf Schichten aus dem Skript, mit denselben Namen wie dort (Abschnitt 1.1 bis 1.3)
ist = np.array([400, 200, 100, 400, 100])
ausschuss = np.array([8, 8, 1, 12, 1])
quoten = ausschuss / ist                      # elementweise, keine Schleife
print(quoten, quoten.dtype, quoten.shape)

maske = quoten > 0.015
print(maske)
print(ist[maske], maske.sum(), maske.mean())

# Dieselben zwei Spalten der echten Datei als Arrays, benannt wie im Skript (Abschnitt 1.4)
ist_alle = np.array([int(z["ist_stueck"]) for z in zeilen])
ausschuss_alle = np.array([int(z["ausschuss_stueck"]) for z in zeilen])
print(len(ist_alle), "Werte, die ersten fünf:", ist_alle[:5], ist_alle.dtype)

Wie viel schneller ist die vektorisierte Rechnung? Die Zelle nimmt die Daten hundertmal hintereinander (578.800 Schichten) und misst beide Wege. Das Ergebnis hängt von deinem Rechner ab.

In [ ]:
import time

viel_ist, viel_aus = np.tile(ist_alle, 100), np.tile(ausschuss_alle, 100)
liste_ist, liste_aus = viel_ist.tolist(), viel_aus.tolist()

start = time.perf_counter()
q_schleife = [a / s for a, s in zip(liste_aus, liste_ist)]
mitte = time.perf_counter()
q_array = viel_aus / viel_ist
ende = time.perf_counter()

t_schleife, t_array = mitte - start, max(ende - mitte, 1e-6)
print(f"Schleife {t_schleife * 1000:.0f} ms, Array {t_array * 1000:.1f} ms, also etwa {t_schleife / t_array:.0f}-mal so schnell")
print("Gleiches Ergebnis:", np.allclose(q_schleife, q_array))

### TODO 1 · Ausschussquote mit NumPy
Berechne mit den Arrays `ist_alle` und `ausschuss_alle` ohne Schleife:
1. `quoten_alle`: die Ausschussquote jeder Schicht als Array mit 5.788 Werten,
2. `anteil_ueber_3`: den Anteil der Schichten mit einer Quote über 3 % (als Zahl zwischen 0 und 1, mit einer Maske),
3. `quote_gesamt`: den gesamten Ausschuss geteilt durch alle gefertigten Stück.

In [ ]:
# TODO 1: Ersetze None durch deine Rechnung mit den Arrays ist_alle und ausschuss_alle.
quoten_alle = None
anteil_ueber_3 = None
quote_gesamt = None

print(quoten_alle, anteil_ueber_3, quote_gesamt)

In [ ]:
check("TODO 1a quoten_alle", lambda: isinstance(quoten_alle, np.ndarray) and quoten_alle.shape == (5788,) and math.isclose(quoten_alle[0], 2 / 314))
check("TODO 1b anteil_ueber_3", lambda: math.isclose(anteil_ueber_3, 0.211990, abs_tol=1e-6))
check("TODO 1c quote_gesamt", lambda: math.isclose(quote_gesamt, 0.021488, abs_tol=1e-6))

In [ ]:
if quoten_alle is None or quote_gesamt is None:
    print("⏳ Erst TODO 1 lösen.")
else:
    print(f"Mittelwert der 5.788 Schichtquoten: {quoten_alle.mean():.2%}")
    print(f"Gesamter Ausschuss durch alle Stück: {quote_gesamt:.2%}")

Zwei Zahlen für „die Ausschussquote“, 2,02 % und 2,15 %. Welche davon die Ausschussquote des Werks ist und woher der Unterschied kommt, klärt Abschnitt 4.

## 2 · DataFrame, Series und Datentypen
*Lerneinheit 2 · Skript Abschnitt 2*

Ein **DataFrame** ist eine Tabelle aus Spalten. Jede Spalte ist eine **Series**: ein Array mit einem Datentyp plus ein **Index**, der die Zeilen beschriftet. `pd.read_csv` liest eine CSV-Datei als DataFrame und rät für jede Spalte einen Datentyp. Prüfe ihn immer mit `.dtypes`.

| Datentyp | Beispiel in den Daten | Bemerkung |
|---|---|---|
| `int64` | `ist_stueck` | Ganzzahl |
| `float64` | `auslastung` | Gleitkommazahl, auch für fehlende Werte |
| Text | `datum` und `schicht` nach dem Einlesen | pandas 3 zeigt `str`, pandas 2 im Browser `object` |
| `bool` | `behoben` in `stoerungen.csv` | Wahrheitswert |
| `datetime64` | `datum` nach der Umwandlung | Zeitstempel |
| `category` | `schicht` nach der Umwandlung | feste Werte, auf Wunsch mit Ordnung |

Fehlende Werte stehen als `NaN` („not a number“) in der Tabelle. Die Nordwerk-Dateien haben keine, Modul 3 behandelt sie ausführlich.

In [ ]:
import pandas as pd

roh = pd.read_csv("daten/produktion.csv")
print(roh.shape)
print(roh.dtypes)
roh.head()

In [ ]:
ist_s = roh["ist_stueck"]                      # eine Spalte ist eine Series
print(type(ist_s).__name__, len(ist_s), ist_s.sum())
print("Gesamtquote in einer Zeile:", roh["ausschuss_stueck"].sum() / roh["ist_stueck"].sum())
print("datum ist Text:", pd.api.types.is_string_dtype(roh["datum"]))
print("datum ist ein Zeitstempel:", pd.api.types.is_datetime64_any_dtype(roh["datum"]))

`pd.to_datetime(spalte, format="%Y-%m-%d")` wandelt Text in Zeitstempel um. Mit festem Format fallen Tippfehler sofort auf. Eine **geordnete Kategorie** legst du mit `pd.CategoricalDtype([...], ordered=True)` an und wendest sie mit `.astype(...)` an. Ein Wert, der nicht in der Liste steht, wird dabei zu einem fehlenden Wert, prüfe also danach. Ein kleines Beispiel mit Prioritäten:

In [ ]:
datumstexte = pd.Series(["2025-03-03", "2025-03-04", "2025-03-10"])
zeitpunkte = pd.to_datetime(datumstexte, format="%Y-%m-%d")
print(zeitpunkte.dt.dayofweek.tolist(), "(0 = Montag)")
print((zeitpunkte.max() - zeitpunkte.min()).days, "Tage")

PRIO_TYP = pd.CategoricalDtype(["niedrig", "mittel", "hoch"], ordered=True)
prio = pd.Series(["hoch", "niedrig", "mittel", "hoch"]).astype(PRIO_TYP)
print(prio.sort_values().tolist())
print((prio >= "mittel").tolist())

### TODO 2 · Laden mit den richtigen Datentypen
1. Lege `SCHICHT_TYP` als geordnete Kategorie mit der Reihenfolge Früh < Spät < Nacht an.
2. Wandle in `produktion` die Spalte `datum` in einen Zeitstempel um und die Spalte `schicht` in `SCHICHT_TYP`.

In [ ]:
produktion = pd.read_csv("daten/produktion.csv")

# TODO 2: Lege SCHICHT_TYP an und wandle datum und schicht um (je eine Zeile).
SCHICHT_TYP = None

print(produktion.dtypes)

In [ ]:
check("TODO 2a SCHICHT_TYP", lambda: isinstance(SCHICHT_TYP, pd.CategoricalDtype) and SCHICHT_TYP.ordered
      and list(SCHICHT_TYP.categories) == ["Früh", "Spät", "Nacht"])
check("TODO 2b datum ist ein Zeitstempel", lambda: pd.api.types.is_datetime64_any_dtype(produktion["datum"])
      and produktion["datum"].min() == pd.Timestamp("2024-10-01"))
check("TODO 2c schicht ist eine geordnete Kategorie", lambda: isinstance(produktion["schicht"].dtype, pd.CategoricalDtype)
      and produktion["schicht"].cat.ordered and list(produktion["schicht"].cat.categories) == ["Früh", "Spät", "Nacht"]
      and produktion["schicht"].notna().all())

Die Störungsmeldungen bekommen dieselben Datentypen. Die beiden Stammdatentabellen brauchen keine Umwandlung.

In [ ]:
stoerungen = pd.read_csv("daten/stoerungen.csv")
stoerungen["datum"] = pd.to_datetime(stoerungen["datum"], format="%Y-%m-%d")
if isinstance(SCHICHT_TYP, pd.CategoricalDtype):
    stoerungen["schicht"] = stoerungen["schicht"].astype(SCHICHT_TYP)
else:
    print("⏳ Erst TODO 2 lösen. Bis dahin bleibt schicht in stoerungen Text.")
maschinen = pd.read_csv("daten/maschinen.csv")
linien = pd.read_csv("daten/linien.csv")

for titel, tab in [("produktion", produktion), ("stoerungen", stoerungen), ("maschinen", maschinen), ("linien", linien)]:
    print(f"{titel:<11} {tab.shape[0]:>6} Zeilen, {tab.shape[1]:>2} Spalten")

Die kleine Tabelle `fuenf` aus dem Skript entsteht hier genauso wie dort (Abschnitte 2.1, 2.4 und 2.5): mit `datum` als Zeitstempel und `schicht` als geordneter Kategorie. Dieselben Ausdrücke wie im Skript und in den Lerneinheiten liefern dann dieselben Ergebnisse.

In [ ]:
fuenf = pd.DataFrame({
    "datum": ["2025-03-03", "2025-03-03", "2025-03-03", "2025-03-04", "2025-03-04"],
    "linie": [1, 1, 4, 1, 4],
    "schicht": ["Früh", "Nacht", "Früh", "Spät", "Nacht"],
    "soll_stueck": [400, 250, 125, 400, 100],
    "ist_stueck": [400, 200, 100, 400, 100],
    "ausschuss_stueck": [8, 8, 1, 12, 1],
})
fuenf["datum"] = pd.to_datetime(fuenf["datum"], format="%Y-%m-%d")
if isinstance(SCHICHT_TYP, pd.CategoricalDtype):
    fuenf["schicht"] = fuenf["schicht"].astype(SCHICHT_TYP)
else:
    print("⏳ Erst TODO 2 lösen. Bis dahin bleibt schicht in fuenf Text.")
print(fuenf)

Eine neue Spalte entsteht durch Zuweisung, `df["neu"] = ...`. Rechnungen mit Spalten wirken elementweise wie bei Arrays, etwa die Gutteile je Schicht. Mit `fuenf["gut_stueck"] = ...` würde daraus eine neue Spalte, hier bleibt `fuenf` wie im Skript.

In [ ]:
print((fuenf["ist_stueck"] - fuenf["ausschuss_stueck"]).tolist())

### TODO 3 · Neue Spalten: Quote und Planerfüllung
Lege in `produktion` zwei Spalten an:
- `quote`: die Ausschussquote `ausschuss_stueck / ist_stueck`,
- `planerfuellung`: gefertigte durch geplante Stück, `ist_stueck / soll_stueck`.

In [ ]:
# TODO 3: Lege die Spalten quote und planerfuellung an (je eine Zeile).
pass

print(produktion.head(3))

In [ ]:
check("TODO 3a quote", lambda: math.isclose(produktion["quote"].mean(), 0.020178, abs_tol=1e-6))
check("TODO 3b planerfuellung", lambda: math.isclose(produktion["planerfuellung"].mean(), 0.874600, abs_tol=1e-6))

## 3 · Auswählen und filtern: loc, iloc und Masken
*Lerneinheit 3 · Skript Abschnitt 3*

- `df["spalte"]` liefert eine Spalte, `df[["a", "b"]]` mehrere.
- `df.loc[zeilen, spalten]` wählt über **Beschriftungen**, also Index und Spaltennamen. Ein Bereich `a:b` schließt `b` ein.
- `df.iloc[zeilen, spalten]` wählt über **Positionen** 0, 1, 2 und so weiter. Ein Bereich `a:b` schließt `b` aus, wie bei Listen.
- Eine Maske ist eine boolesche Series. Bedingungen verknüpfst du mit `&` (und), `|` (oder) und `~` (nicht). Jede Bedingung braucht eigene Klammern, weil `&` und `|` stärker binden als `==` oder `<`. Ohne Klammern rechnet pandas manchmal ohne jede Fehlermeldung etwas anderes (Skript, Stolperstein in Abschnitt 3). Prüfe deshalb immer die Zeilenzahl.
- Willst du eine Teilmenge weiterbearbeiten, trenne sie mit `.copy()` ab, und ändere Werte immer mit einem einzigen `df.loc[maske, "spalte"] = wert`.

In [ ]:
# Wie im Skript, Abschnitt 2.6: fuenf bekommt Quote und Planerfüllung
fuenf["quote"] = fuenf["ausschuss_stueck"] / fuenf["ist_stueck"]
fuenf["planerfuellung"] = fuenf["ist_stueck"] / fuenf["soll_stueck"]

print(fuenf.loc[1:3, ["linie", "schicht"]])     # Beschriftungen 1, 2 und 3
print(fuenf.iloc[1:3, [1, 2]])                  # Positionen 1 und 2

linie1 = fuenf[fuenf["linie"] == 1]             # behält die Beschriftungen 0, 1 und 3
print(linie1.index.tolist())
print(linie1.loc[3, "ist_stueck"], linie1.iloc[2, 4])

In [ ]:
# Masken an den fünf Schichten, wie im Skript
print(fuenf[(fuenf["linie"] == 1) & (fuenf["schicht"] != "Früh")].index.tolist())
print(fuenf[(fuenf["quote"] > 0.025) | (fuenf["planerfuellung"] < 1)].index.tolist())
print(fuenf[~(fuenf["linie"] == 1)].index.tolist())

nacht_l2 = (produktion["linie"] == 2) & (produktion["schicht"] == "Nacht")
print(nacht_l2.sum(), "Nachtschichten auf Linie 2,", (~nacht_l2).sum(), "andere Schichten")

# Kurzformen
print(produktion["linie"].isin([1, 2]).sum(), "Schichten auf Linie 1 oder 2")
print(produktion["auslastung"].between(0.9, 0.95).sum(), "Schichten mit Auslastung von 0,90 bis 0,95")
print(len(produktion.query("linie == 3 and ist_stueck > 280")), "Schichten auf Linie 3 mit mehr als 280 Stück")

### TODO 4 · Auffällige Schichten auswählen
Die Werksleitung möchte alle Schichten sehen, die
- auf Linie 1 oder 2 liefen (nimm `isin`),
- am 1. Januar 2026 oder später stattfanden,
- **und** entweder eine Planerfüllung unter 0,8 **oder** eine Ausschussquote über 0,05 hatten.

Speichere die Maske in `maske` und die ausgewählten Zeilen mit den Spalten `datum`, `linie`, `schicht`, `planerfuellung` und `quote` in `auffaellig`, ausgewählt mit `.loc`. Einen Zeitstempel kannst du direkt mit einem Text wie `"2026-01-01"` vergleichen.

In [ ]:
# TODO 4: Maske aus drei Bedingungen, dann Auswahl mit .loc
maske = None
auffaellig = None

if auffaellig is not None:
    print(len(auffaellig), "Schichten")
    print(auffaellig.head())

In [ ]:
check("TODO 4a Anzahl", lambda: len(auffaellig) == 244)
check("TODO 4b Spalten", lambda: list(auffaellig.columns) == ["datum", "linie", "schicht", "planerfuellung", "quote"])
check("TODO 4c nur Linie 1 und 2 ab 2026", lambda: set(auffaellig["linie"]) == {1, 2}
      and auffaellig["datum"].min() >= pd.Timestamp("2026-01-01"))

## 4 · Split-Apply-Combine: gruppieren und aggregieren
*Lerneinheit 4 · Skript Abschnitt 4*

`groupby` teilt die Tabelle nach einem Schlüssel in Gruppen (split), wendet auf jede Gruppe eine Funktion an (apply) und fügt die Ergebnisse zu einer neuen Tabelle zusammen (combine). Das Akkumulator-Muster „Stillstand je Maschine“ aus Modul 1 wird so zu einer Zeile:

In [ ]:
stillstand_je_maschine = stoerungen.groupby("maschine")["stillstand_min"].sum()
print(stillstand_je_maschine.sort_values(ascending=False).head())

Mit **benannter Aggregation** berechnest du mehrere Kennzahlen auf einmal: `agg(name=(spalte, funktion))`. Gruppierst du nach einer Kategorie wie `schicht`, gib `observed=True` an. Sonst warnt pandas 2 im Browser, und Kategorien ohne Zeilen erscheinen als leere Gruppen.

In [ ]:
if "planerfuellung" not in produktion.columns:
    print("⏳ Erst TODO 3 lösen.")
else:
    kennzahlen = produktion.groupby("linie").agg(          # wie im Skript, Abschnitt 4.2
        schichten=("ist_stueck", "size"),
        ist_mittel=("ist_stueck", "mean"),
        planerfuellung=("planerfuellung", "mean"),
        personal=("personal", "mean"),
    )
    print(kennzahlen.round(3))

**Quoten richtig aggregieren.** Die Ausschussquote einer Gruppe ist der gesamte Ausschuss geteilt durch alle gefertigten Stück:

$$q_{\text{Gruppe}} = \frac{\sum_i a_i}{\sum_i s_i} = \sum_i w_i \, \frac{a_i}{s_i}, \qquad w_i = \frac{s_i}{\sum_j s_j}.$$

Diese **Quote der Summen** ist ein gewichtetes Mittel der Schichtquoten, jede Schicht zählt mit ihrem Anteil an der Stückzahl. Das **Mittel der Quoten** $\frac{1}{n}\sum_i a_i/s_i$ gibt dagegen jeder Schicht dasselbe Gewicht. Bei gleichen Stückzahlen stimmen beide überein.

### TODO 5 · Ausschussquote je Linie und Schicht
Gruppiere `produktion` nach `linie` und `schicht` (mit `observed=True`) und berechne mit benannter Aggregation die Spalten
- `ist`: Summe von `ist_stueck`,
- `ausschuss`: Summe von `ausschuss_stueck`,
- `mittel_der_quoten`: Mittelwert der Spalte `quote`.

Ergänze danach die Spalte `quote_der_summen` als `ausschuss / ist`. Speichere das Ergebnis in `je_ls`.

In [ ]:
# TODO 5: benannte Aggregation, danach die Spalte quote_der_summen
je_ls = None

print(je_ls)

In [ ]:
check("TODO 5a zwölf Gruppen", lambda: len(je_ls) == 12 and list(je_ls.index.names) == ["linie", "schicht"])
check("TODO 5b Summen", lambda: je_ls["ist"].sum() == 1217169 and je_ls["ausschuss"].sum() == 26154)
check("TODO 5c quote_der_summen", lambda: math.isclose(je_ls.loc[(3, "Spät"), "quote_der_summen"], 0.016175, abs_tol=1e-6))
check("TODO 5d mittel_der_quoten", lambda: math.isclose(je_ls.loc[(3, "Spät"), "mittel_der_quoten"], 0.015985, abs_tol=1e-6))

In [ ]:
if je_ls is None or "quote_der_summen" not in je_ls.columns or "quote" not in produktion.columns:
    print("⏳ Erst TODO 3 und TODO 5 lösen.")
else:
    abstand = (je_ls["quote_der_summen"] - je_ls["mittel_der_quoten"]).abs().max()
    print(f"Größter Abstand innerhalb einer Linie und Schicht: {abstand * 100:.3f} Prozentpunkte")
    je_linie = produktion.groupby("linie").agg(ist=("ist_stueck", "sum"), ausschuss=("ausschuss_stueck", "sum"),
                                              mittel_der_quoten=("quote", "mean"))
    je_linie["quote_der_summen"] = je_linie["ausschuss"] / je_linie["ist"]
    print(je_linie.round(4))
    gesamt = produktion["ausschuss_stueck"].sum() / produktion["ist_stueck"].sum()
    print(f"Werk gesamt: Mittel der Quoten {produktion['quote'].mean():.2%}, Quote der Summen {gesamt:.2%}")

Innerhalb einer Linie und Schicht liegen beide Maße fast gleich, weil die Stückzahlen dort ähnlich sind. Über das ganze Werk weichen sie deutlich ab: Linie 1 fertigt viel und hat eine hohe Quote, Linie 4 fertigt wenig und hat eine niedrige. Der Anteil des Ausschusses an allen gefertigten Teilen ist die Quote der Summen, 2,15 %. Wie groß der Abstand genau ist, zeigt das Skript in Abschnitt 4.4 und Bonus B1.

`transform` rechnet je Gruppe einen Wert aus und schreibt ihn in jede Zeile der Gruppe zurück. Die Tabelle behält ihre Zeilenzahl, und Anteile an der Gruppe werden zu einer Division:

In [ ]:
fuenf["anteil_linie"] = fuenf["ist_stueck"] / fuenf.groupby("linie")["ist_stueck"].transform("sum")
print(fuenf[["linie", "ist_stueck", "anteil_linie"]])

### TODO 6 · Anteil an der Gruppe mit transform
Welchen Teil der Tagesmenge einer Linie liefert jede Schicht? Lege in `produktion` die Spalte `anteil_tag` an: `ist_stueck` geteilt durch die Summe von `ist_stueck` derselben Linie am selben Tag. Berechne dann `anteil_nacht`, den mittleren Anteil der Nachtschichten.

In [ ]:
# TODO 6: Spalte anteil_tag mit groupby(...).transform("sum"), danach anteil_nacht
anteil_nacht = None

print(anteil_nacht)

In [ ]:
check("TODO 6a Anteile summieren sich je Tag und Linie zu 1",
      lambda: np.allclose(produktion.groupby(["datum", "linie"])["anteil_tag"].sum(), 1))
check("TODO 6b anteil_nacht", lambda: math.isclose(anteil_nacht, 0.308495, abs_tol=1e-6))

## 5 · Tabellen verknüpfen: Schlüssel, Join-Arten, Kardinalität
*Lerneinheit 5 · Skript Abschnitt 5*

`links.merge(rechts, on="schluessel", how=...)` kombiniert jede Zeile links mit jeder Zeile rechts, die denselben Schlüssel hat.
- `how="inner"`: nur Schlüssel, die in beiden Tabellen vorkommen,
- `how="left"`: alle Zeilen von links, ohne Partner mit `NaN` in den rechten Spalten,
- `how="right"`: entsprechend alle Zeilen von rechts,
- `how="outer"`: alle Schlüssel aus beiden Tabellen.

Beim inner join hat das Ergebnis $\sum_k n_L(k)\, n_R(k)$ Zeilen, summiert über alle Schlüsselwerte $k$. Dabei zählen $n_L(k)$ und $n_R(k)$ die Zeilen mit dem Schlüssel $k$ links und rechts. Ist der Schlüssel rechts eindeutig (Kardinalität m:1), wächst die Zeilenzahl nicht. Sind beide Seiten mehrdeutig (m:n), explodiert sie. Zwei Absicherungen:
- `validate="many_to_one"` (ebenso `"one_to_one"` und `"one_to_many"`, kurz `"m:1"`, `"1:1"` und `"1:m"`) bricht mit einem Fehler ab, wenn die Kardinalität nicht stimmt. `"many_to_many"` (`"m:m"`) prüft nichts und hält nur fest, dass m:n gewollt ist,
- `indicator=True` ergänzt die Spalte `_merge` mit `both`, `left_only` oder `right_only`.

In [ ]:
meldungen = pd.DataFrame({"id": ["S-1", "S-2", "S-3", "S-4"], "maschine": ["M-03", "M-05", "M-03", "M-99"]})
stamm = pd.DataFrame({"maschine": ["M-03", "M-05", "M-07"], "linie": [1, 3, 4]})
for art in ["inner", "left", "right", "outer"]:
    print(art, len(meldungen.merge(stamm, on="maschine", how=art)), "Zeilen")
print(meldungen.merge(stamm, on="maschine", how="outer", indicator=True))

try:
    meldungen.merge(stamm, on="maschine", validate="one_to_one")
except Exception as fehler:
    print("validate meldet:", type(fehler).__name__, str(fehler).splitlines()[0])

### TODO 7 · Störungen mit Maschinen verknüpfen
`stoerungen` kennt die Maschine, aber nicht die Linie. Die steht in `maschinen`.
1. Hänge an `stoerungen` die Spalten `linie` und `typ` aus `maschinen` an: left join über `maschine` mit `validate="many_to_one"`. Nimm von `maschinen` nur die Spalten `maschine`, `linie` und `typ`, wie im Skript (Abschnitt 5.4). Speichere das Ergebnis als `st_m`.
2. Berechne `stillstand_linie`, den gesamten Stillstand je Linie in **Stunden**, als Series mit der Linie als Index.

In [ ]:
# TODO 7
st_m = None
stillstand_linie = None

print(stillstand_linie)

In [ ]:
check("TODO 7a Zeilenzahl bleibt, jede Meldung hat eine Linie", lambda: len(st_m) == 14263 and st_m["linie"].notna().all())
check("TODO 7b stillstand_linie", lambda: math.isclose(stillstand_linie.loc[1], 2870.0, abs_tol=0.01)
      and math.isclose(stillstand_linie.sum(), 9105.33, abs_tol=0.01))

Jetzt der typische Fehler. Du willst wissen, wie Stillstand und Stückzahl zusammenhängen, und hängst die Störungsmeldungen über Tag und Linie an die Produktionszeilen. Je Tag und Linie gibt es links bis zu drei Schichten und rechts mehrere Meldungen. Das ist m:n: Jede Produktionszeile erscheint so oft, wie es an diesem Tag Meldungen auf ihrer Linie gibt, und jede Summe über `ist_stueck` wird zu groß. Im Kleinen:

In [ ]:
prod_klein = pd.DataFrame({"datum": ["2025-03-03"] * 3, "linie": [1, 1, 1],
                           "schicht": ["Früh", "Spät", "Nacht"], "ist_stueck": [400, 400, 300]})
stoer_klein = pd.DataFrame({"datum": ["2025-03-03"] * 2, "linie": [1, 1], "stillstand_min": [30, 45]})
falsch_klein = prod_klein.merge(stoer_klein, on=["datum", "linie"])
print(len(falsch_klein), "Zeilen statt 3, Summe ist_stueck", falsch_klein["ist_stueck"].sum(), "statt", prod_klein["ist_stueck"].sum())

### TODO 8 · Die m:n-Falle erkennen und lösen
**a) Erkennen.** Sag die Zeilenzahl des inner joins von `produktion` und `st_m` über `["datum", "linie"]` mit der Formel voraus, bevor du ihn ausführst:
`n_links = produktion.groupby(["datum", "linie"]).size()` zählt $n_L(k)$, `n_rechts` entsprechend für `st_m`. Das Produkt `n_links * n_rechts` ordnet die Zähler über den Index einander zu, Schlüssel ohne Partner ergeben `NaN`, und `sum` überspringt sie. `zeilen_formel` ist diese Summe als ganze Zahl. Führe danach den Join aus und speichere ihn als `falsch`.

**b) Lösen.** Erst aggregieren, dann verknüpfen:
1. `st_schicht`: Störungen je `datum`, `linie` und `schicht` mit den Spalten `stoerungen` (Anzahl der Meldungen) und `stillstand_min` (Summe), danach `.reset_index()`.
2. `prod_st`: left join von `produktion` mit `st_schicht` über alle drei Schlüssel, mit `validate="one_to_one"`.
3. Schichten ohne Störung haben jetzt `NaN`. Setze sie in beiden Spalten auf 0.

In [ ]:
# TODO 8a: Zeilenzahl vorher mit der Formel, danach der Join
zeilen_formel = None
falsch = None

# TODO 8b: erst aggregieren, dann left join, fehlende Werte als 0
st_schicht = None
prod_st = None

if falsch is not None:
    print("Formel:", zeilen_formel, "| Join:", len(falsch))
    print("ist_stueck richtig:", produktion["ist_stueck"].sum(), "| nach dem Join:", falsch["ist_stueck"].sum())
if prod_st is not None:
    print("Zeilen nach dem richtigen Join:", len(prod_st))

In [ ]:
check("TODO 8a Formel und Join stimmen überein", lambda: zeilen_formel == len(falsch) == 42319)
check("TODO 8a Summe nach dem falschen Join", lambda: falsch["ist_stueck"].sum() == 8914442)
check("TODO 8b st_schicht", lambda: len(st_schicht) == 5268 and st_schicht["stillstand_min"].sum() == 546320)
check("TODO 8b Zeilenzahl und Stückzahl bleiben", lambda: len(prod_st) == 5788 and prod_st["ist_stueck"].sum() == 1217169)
check("TODO 8b fehlende Werte als 0", lambda: prod_st["stoerungen"].isna().sum() == 0
      and (prod_st["stoerungen"] == 0).sum() == 520 and prod_st["stillstand_min"].sum() == 546320)

In [ ]:
if prod_st is None or "stillstand_min" not in prod_st.columns or "planerfuellung" not in prod_st.columns:
    print("⏳ Erst TODO 3 und TODO 8 lösen.")
else:
    ohne = prod_st["stoerungen"] == 0
    viel = prod_st["stillstand_min"] > 240
    print(f"Planerfüllung in {ohne.sum()} Schichten ohne Störung:            {prod_st.loc[ohne, 'planerfuellung'].mean():.1%}")
    print(f"Planerfüllung in {viel.sum()} Schichten mit mehr als 4 h Stillstand: {prod_st.loc[viel, 'planerfuellung'].mean():.1%}")
    print(f"Korrelation von Stillstand und Planerfüllung: {prod_st['stillstand_min'].corr(prod_st['planerfuellung']):.2f}")

Erst durch den richtigen Join wird die Frage beantwortbar: Je mehr Stillstand eine Schicht hat, desto weiter bleibt sie hinter dem Plan zurück. Der Zusammenhang ist deutlich, aber nicht vollständig, denn auch ohne Störungsmeldung gehen Stück verloren, etwa beim Rüsten oder wenn Personal oder Material fehlen. Auch Schichten ohne Störung erreichen deshalb im Mittel nicht ganz 100 %.

## 6 · Tidy Data: breit und lang, pivot und melt
*Lerneinheit 6 · Skript Abschnitt 6*

Eine Tabelle ist **tidy** (Wickham 2014), wenn
1. jede Variable eine Spalte ist,
2. jede Beobachtung eine Zeile ist,
3. jede Beobachtungseinheit eine eigene Tabelle bildet.

Der Wochenbericht verletzt die ersten beiden Regeln: Die Spaltennamen `L1_Früh` bis `L4_Nacht` enthalten die Werte zweier Variablen, Linie und Schicht, und eine Zeile enthält zwölf Beobachtungen. `melt` macht aus den zwölf Wertspalten zwei, eine mit dem alten Spaltennamen und eine mit dem Wert. `pivot_table` geht den Weg zurück.

In [ ]:
breit = pd.read_csv("daten/wochenbericht_breit.csv")
print(breit.shape)
print(breit.iloc[:3, :5])

mini = pd.DataFrame({"kw": ["2025-KW10", "2025-KW11"], "L1_Früh": [1600, 1580], "L1_Nacht": [1350, 1400]})
mini_lang = mini.melt(id_vars="kw", var_name="spalte", value_name="ist_stueck")
print(mini_lang)

# Wie im Skript, Abschnitte 6.3 und 6.4: Spaltennamen zerlegen, dann zurück in die Breite
teile = mini_lang["spalte"].str.split("_", expand=True)
print(teile)
if isinstance(SCHICHT_TYP, pd.CategoricalDtype):
    mini_lang["linie"] = teile[0].str[1:].astype("int64")
    mini_lang["schicht"] = teile[1].astype(SCHICHT_TYP)
    mini_lang = mini_lang.drop(columns="spalte")
    print(mini_lang)
    print(mini_lang.pivot(index="kw", columns="schicht", values="ist_stueck"))
else:
    print("⏳ Erst TODO 2 lösen.")

### TODO 9 · Wochenbericht in die lange Form
1. `lang`: Bring `breit` mit `melt` in die lange Form (`id_vars="kw"`, `var_name="spalte"`, `value_name="ist_stueck"`).
2. Zerlege `spalte` mit `.str.split("_", expand=True)`. Aus dem ersten Teil wird die Ganzzahl-Spalte `linie` (`.str[1:]` schneidet das `L` ab, `.astype("int64")` wandelt um), aus dem zweiten die Spalte `schicht` mit `SCHICHT_TYP`. Entferne danach `spalte`.
3. Gegenprobe: `abweichung` ist die Summe von `ist_stueck` je Linie und Schicht in `lang` minus dieselbe Summe in `produktion`. Sie sollte überall 0 sein.

In [ ]:
# TODO 9a: melt
lang = None

# TODO 9b: spalte in linie und schicht zerlegen, spalte entfernen

# TODO 9c: Gegenprobe gegen produktion
abweichung = None

if lang is not None:
    print(lang.head())
print(abweichung)

In [ ]:
check("TODO 9a lange Form", lambda: len(lang) == 1212 and set(lang.columns) == {"kw", "ist_stueck", "linie", "schicht"})
check("TODO 9b Datentypen", lambda: pd.api.types.is_integer_dtype(lang["linie"])
      and isinstance(lang["schicht"].dtype, pd.CategoricalDtype) and lang["schicht"].notna().all())
check("TODO 9c Gegenprobe", lambda: len(abweichung) == 12 and (abweichung == 0).all())

In [ ]:
if lang is None or "linie" not in lang.columns or "schicht" not in lang.columns:
    print("⏳ Erst TODO 9 lösen.")
else:
    zurueck = lang.pivot_table(index="kw", columns=["linie", "schicht"], values="ist_stueck", aggfunc="sum", observed=True)
    print(zurueck.shape)
    print(zurueck.iloc[:3, :6])
    print("Zurück in der Breite gleich wie der Bericht:", (zurueck.to_numpy() == breit.drop(columns="kw").to_numpy()).all())

## 7 · Zeit: Zeitstempel, Zeiträume, resample
*Lerneinheit 7 · Skript Abschnitt 7*

- Der Accessor `.dt` liefert Teile eines Zeitstempels: `.dt.dayofweek` (0 = Montag), `.dt.month`, `.dt.isocalendar()` mit ISO-Jahr und ISO-Kalenderwoche. `.dt.strftime("%G-KW%V")` erzeugt Wochentexte wie im Bericht.
- Zeiträume filterst du mit Vergleichen oder mit `.between("2025-12-01", "2026-02-28")`.
- `resample("W")` braucht einen Zeitstempel als Index und fasst zu Kalenderwochen zusammen. Beschriftet wird jede Woche mit ihrem **Sonntag**.
- `rolling(4).mean()` bildet einen gleitenden Mittelwert über vier aufeinanderfolgende Werte. Mit `min_periods=1` genügt ein vorhandener Wert im Fenster.

**Lücken:** In den Kalenderwochen 33 und 34 ruht das Werk (Werksferien), Feiertage und die Tage vom 24.12. bis 1.1. machen andere Wochen kürzer. `resample(...).sum()` liefert für eine Woche ohne Zeilen 0, `mean()` liefert `NaN`. Welche Wahl stimmt, hängt von der Frage ab: „Wie viel wurde gefertigt?“ verträgt die 0, „Wie viel fertigt das Werk in einer Betriebswoche?“ nicht.

In [ ]:
if not pd.api.types.is_datetime64_any_dtype(produktion["datum"]):
    print("⏳ Erst TODO 2 lösen.")
else:
    tage_nw = produktion.drop_duplicates("datum")["datum"]
    print("Produktionstage je Wochentag (0 = Montag):", tage_nw.dt.dayofweek.value_counts().sort_index().to_dict())
    kw_text = produktion["datum"].dt.strftime("%G-KW%V")
    je_kw = produktion.groupby(kw_text)["ist_stueck"].sum()
    print(len(je_kw), "Wochen mit Produktion, gleiche Summen wie im Bericht:",
          (je_kw.to_numpy() == breit.drop(columns="kw").sum(axis=1).to_numpy()).all())
    winter = produktion[produktion["datum"].between("2025-12-01", "2026-02-28")]
    print("Schichten im Winter 2025/26:", len(winter))

In [ ]:
reihe = pd.Series([10, 12, 11, 13, 9, 14, 15, 12], index=pd.to_datetime([
    "2025-03-03", "2025-03-04", "2025-03-05", "2025-03-06", "2025-03-07",
    "2025-03-17", "2025-03-18", "2025-03-19"]))
print(reihe.resample("W").sum())               # die leere Woche wird 0
print(reihe.resample("W").sum(min_count=1))    # die leere Woche wird NaN

### TODO 10 · Wochenwerte und gleitender Mittelwert
1. `woche`: Wochensummen von `ist_stueck` und `ausschuss_stueck`. Setze dazu `datum` als Index und nimm `resample("W").sum()`.
2. `woche["ist_4w"]`: gleitender Mittelwert von `ist_stueck` über 4 Wochen. Setze vorher die Ferienwochen mit 0 Stück auf `NaN` (`.where(woche["ist_stueck"] > 0)`) und nimm `rolling(4, min_periods=1)`, damit nur Wochen mit Produktion zählen.
3. `woche["quote_4w"]`: Ausschussquote der jeweils letzten vier Wochen als Quote der Summen, also gleitende Summe des Ausschusses geteilt durch gleitende Summe der Stückzahl (`rolling(4).sum()`).

In [ ]:
# TODO 10
woche = None

if woche is not None:
    print(woche.loc["2025-07-27":"2025-09-21"])

In [ ]:
check("TODO 10a woche", lambda: len(woche) == 105 and woche["ist_stueck"].sum() == 1217169 and (woche["ist_stueck"] == 0).sum() == 4)
check("TODO 10b ist_4w ohne Ferienwochen", lambda: math.isclose(woche.loc[pd.Timestamp("2025-08-24"), "ist_4w"], 11892.0)
      and woche["ist_4w"].notna().all())
check("TODO 10c quote_4w", lambda: math.isclose(woche.loc[pd.Timestamp("2025-06-22"), "quote_4w"], 0.025106, abs_tol=1e-6)
      and woche["quote_4w"].isna().sum() == 3)

In [ ]:
if woche is None or "ist_4w" not in woche.columns or "quote_4w" not in woche.columns:
    print("⏳ Erst TODO 10 lösen.")
else:
    ax = woche[["ist_stueck", "ist_4w"]].plot(figsize=(9, 3.5), title="Gefertigte Stück je Woche, Nordwerk GmbH")
    ax.set_xlabel("Woche (beschriftet mit dem Sonntag)")
    ax.set_ylabel("Stück")
    ax.legend(["Wochensumme", "gleitender Mittelwert über 4 Wochen ohne Ferienwochen"])
    print(f"Höchste 4-Wochen-Ausschussquote: {woche['quote_4w'].max():.2%} in der Woche bis {woche['quote_4w'].idxmax():%d.%m.%Y}")

In der Grafik fallen die Werksferien und die Wochen um den Jahreswechsel als Einbrüche der Wochensumme auf. Über die Ferienwochen läuft der gleitende Mittelwert hinweg. Die kurzen Wochen um den Jahreswechsel ziehen ihn dagegen nach unten, denn sie zählen als Wochen mit Produktion. Die höchste 4-Wochen-Quote fällt in den Sommer. Ob die Jahreszeit dahintersteckt, untersucht TODO 11.

### TODO 11 · Sommer gegen Winter
Wähle mit `.dt.month` die Schichten aus Juni, Juli und August (Sommer) und aus Dezember, Januar und Februar (Winter), beide Jahre zusammen. Berechne für jede Linie die Ausschussquote als Quote der Summen und speichere beides in `vergleich`: eine Tabelle mit der Linie als Index und den Spalten `sommer` und `winter`.

In [ ]:
# TODO 11
vergleich = None

print(vergleich)

In [ ]:
check("TODO 11 vergleich", lambda: len(vergleich) == 4
      and math.isclose(vergleich.loc[1, "sommer"], 0.027640, abs_tol=1e-6)
      and math.isclose(vergleich.loc[1, "winter"], 0.024054, abs_tol=1e-6))

Auf den Linien 1 und 2 liegt die Quote im Sommer deutlich höher, auf den Linien 3 und 4 sogar etwas niedriger. Ob das an der Hallentemperatur liegt, kann diese Tabelle allein nicht entscheiden, denn im Sommer ändert sich mehr als die Temperatur. Dafür brauchst du Methoden, die mehrere Einflüsse zugleich betrachten.

## Bonus

### Bonus B1 · Warum die beiden Quoten auseinanderliegen
Im Skript (Abschnitt 4.4) steht die exakte Beziehung

$$\frac{\sum_i a_i}{\sum_i s_i} - \frac{1}{n}\sum_i q_i = \frac{c_{sq}}{\bar s}, \qquad c_{sq} = \frac{1}{n}\sum_i (s_i - \bar s)(q_i - \bar q).$$

Prüfe sie mit den Arrays `ist_alle` und `quoten_alle` aus TODO 1. Berechne `differenz` (linke Seite) und `kovarianz_term` (rechte Seite). Achtung: `np.cov` teilt durch $n-1$, berechne $c_{sq}$ deshalb mit `np.mean`.

In [ ]:
# Bonus B1
differenz = None
kovarianz_term = None

print(differenz, kovarianz_term)

In [ ]:
check("B1 beide Seiten gleich", lambda: math.isclose(differenz, kovarianz_term, rel_tol=1e-9) and math.isclose(differenz, 0.001310, abs_tol=1e-6))

### Bonus B2 · Ist M-03 auffällig?
In Modul 1 fiel die CNC-Drehmaschine M-03 mit ihrer Spindel auf. Zähle die mechanischen Störungen je Maschine, hänge den Maschinentyp aus `maschinen` an und teile jede Anzahl durch den Mittelwert der Maschinen desselben Typs (`groupby("typ")` mit `transform("mean")`). Speichere das Verhältnis als Series `verhaeltnis` mit der Maschine als Index.

In [ ]:
# Bonus B2
verhaeltnis = None

print(verhaeltnis)

In [ ]:
check("B2 M-03 liegt vorn", lambda: verhaeltnis.idxmax() == "M-03" and math.isclose(verhaeltnis.loc["M-03"], 1.8937, abs_tol=1e-4))

M-03 hat fast doppelt so viele mechanische Störungen wie der Durchschnitt der Zerspanungsmaschinen, sie selbst eingeschlossen. Ob ein Unterschied dieser Größe Zufall sein kann, ist eine Frage für die Statistik.

## Zum Nachdenken

1. In TODO 6 liefert die Nachtschicht im Mittel 31 % der Tagesmenge einer Linie. Für die Frühschicht ergibt sich ein höherer Mittelwert als für die Spätschicht, obwohl beide gleich groß geplant sind. Woran liegt das?
2. Die Werksleitung fragt nach „der mittleren Ausschussquote je Schicht“. Welche Zahl aus Abschnitt 4 gibst du ihr, und wie formulierst du die Frage so, dass keine Missverständnisse bleiben?
3. Nach einem Join hat deine Tabelle mehr Zeilen als vorher. Woran erkennst du, ob das gewollt ist (1:n) oder ein Fehler (m:n)?
4. Für `ist_4w` hast du die Ferienwochen als `NaN` behandelt. Bei welcher Frage wäre 0 die richtige Wahl?

<details><summary>Hinweise zu den Antworten</summary>

1. An 34 Samstagen gibt es nur eine Frühschicht, ihr Anteil ist dann 100 %. Das hebt den Mittelwert der Frühschicht. An Werktagen liegen Früh und Spät fast gleichauf.
2. Die Quote der Summen beantwortet „Welcher Anteil der gefertigten Teile ist Ausschuss?“. Das Mittel der Quoten beantwortet „Wie hoch ist die Quote einer typischen Schicht?“. Nenne die Frage mit, wenn du eine Zahl weitergibst.
3. Prüfe vorher, ob der Schlüssel auf einer Seite eindeutig ist (`is_unique`, `duplicated`), rechne die Zeilenzahl mit $\sum_k n_L(k)\,n_R(k)$ aus und sichere den Join mit `validate` ab.
4. Bei Summen über einen Zeitraum, etwa „Wie viel hat das Werk im August gefertigt?“. Die Ferienwochen tragen dann zu Recht 0 bei.
</details>

## Wie es weitergeht

In diesem Modul waren die Daten sauber. In echten Werksdaten fehlen Werte, Schlüssel kommen doppelt vor, ein Datum steht in drei verschiedenen Formaten, und dieselbe Maschine heißt einmal `M-03` und einmal `M-03 `. **Modul 3** zeigt, wie du solche Probleme findest und behebst, und wie du dieselben Fragen mit **SQL** direkt an eine Datenbank stellst. Vieles kennst du dann schon: Aus `groupby` wird `GROUP BY`, aus `merge` wird `JOIN`.